# M5 Forecasting — Building the Analytical Training Dataset (Database-First, Chunked)

**Project:** Retail-Demand-Forecasting
**Stage:** `04_build_training_dataset` (v2 — redesigned)
**Status:** RESEARCH / EXPERIMENTATION NOTEBOOK — not production code
**Author:** Data Engineering / Analytics Team

---

## Why this notebook exists (and why v1 doesn't work anymore)

The previous version of this notebook read all five tables fully into `pandas` and joined
them with `DataFrame.merge()`. That worked fine against a small sample, but against the real
`sales` table — **~58 million rows** after the wide→long conversion — it failed with a
`MemoryError` while joining `prices` (~6.8 million rows):

```
PostgreSQL
    │
    ▼
Load huge tables into pandas   ←── the problem starts here
    │
    ▼
pandas.merge()
    │
    ▼
pandas.merge()
    │
    ▼
pandas.merge()
    │
    ▼
MemoryError
```

This notebook is a full redesign around a different architecture:

```
PostgreSQL
    │
    ▼
SQL JOIN                        ←── the database does the expensive relational work
    │
    ▼
Database validates the relationship   (row counts, orphan checks — still inside SQL)
    │
    ▼
Chunked retrieval                (pandas.read_sql_query(..., chunksize=...))
    │
    ▼
Small pandas DataFrame            (one chunk at a time, never the full result)
    │
    ▼
Validation / lightweight processing
    │
    ▼
Parquet (written incrementally, part-by-part)
```

**This is still a research notebook.** Its job is to prove this architecture actually works —
end to end, on real (if here, small-scale test) data — before any of it becomes
`src/ml/dataset_builder.py`. Nothing here is optimized code or a reusable module; every step is
inline and inspectable, exactly like the Extract/Transform/Load notebooks before it.


## Section 1 — Why the Previous Pandas Approach Failed

### What actually happened, step by step

1. `sales_df = pd.read_sql("SELECT * FROM sales", engine)` pulled **~58 million rows** into a
   single `pandas` DataFrame — already a large chunk of RAM before any joining even started.
2. Each `.merge()` call doesn't modify a DataFrame in place — it **allocates a brand-new
   DataFrame** for the result, while the original inputs are often still alive in memory
   (Python's garbage collector doesn't necessarily free them immediately, especially if any
   other variable still references them, e.g. for later debugging or display).
3. By the time the notebook reached the `prices` merge (~6.8 million rows on the right side),
   memory held: the original `sales_df` (58M rows), the intermediate `step1`/`step2`/`step3`
   results (each also ~58M rows, now with more columns), *and* `prices_df` — several large
   objects simultaneously, not just one.
4. `step3.merge(prices_df, ...)` needed to allocate **yet another** ~58M-row DataFrame for its
   result, on top of everything already in memory. That allocation is what triggered the
   `MemoryError` — not because that one merge was unusually expensive, but because it was the
   final straw on top of everything already resident.

### The important reframing

**The dataset is not "too big" or "wrong."** ~58 million rows of `float32`/`int32`/short
strings is, in principle, a dataset that fits comfortably in a few gigabytes of RAM *by
itself*. The actual problem is architectural: **we were materializing too much data in Python
process memory at the same time** — multiple full-size copies of a large table, all alive
simultaneously, purely as a side effect of how `pandas.merge()` works.

### Why PostgreSQL is better suited for this specific job

A relational database is built, from the ground up, to perform exactly this kind of operation
— joining large tables — **without** materializing every intermediate result in a single
process's memory:

- PostgreSQL's query planner can choose join algorithms (hash join, merge join, nested loop)
  based on table statistics, indexes, and available resources — `pandas.merge()` always uses
  the same general-purpose approach regardless of data size.
- PostgreSQL can **stream** query results back to the client rather than materializing the
  entire joined result in memory before sending anything — which is exactly what lets us
  retrieve results in chunks (Section 7) instead of all at once.
- Joins execute **inside the database process**, which has its own memory management
  (including spilling to disk for operations too large for RAM) — completely separate from,
  and independent of, the Python process running this notebook.
- Because the raw data already lives in PostgreSQL (that's what the Load-phase notebook
  accomplished), asking the database to join it is not an extra step — it's using the system
  that's already holding the data, for the job it was designed to do.

**The redesigned rule for this notebook:** Python/`pandas` is used for *inspecting*,
*validating*, *light per-chunk processing*, and *saving* — never for holding the full joined
result at once. PostgreSQL does 100% of the actual relational joining.


## Section 2 — Connect to PostgreSQL Using the Project's Existing Configuration

**What we're doing:** Making `src/` importable from this notebook (which lives in
`notebooks/`), then importing the project's existing `DB_CONFIG` from
`src.database.config` and building a SQLAlchemy engine from it.

**Why we're doing it this way:** The task explicitly requires reusing the project's existing
configuration rather than hardcoding credentials here — and the database password contains a
literal `@` character. A manually-built connection string
(`f"postgresql://{user}:{password}@{host}/{db}"`) would **misparse** at that `@`, since `@` is
also the separator between credentials and host in a connection URL. `sqlalchemy.URL.create()`
takes each component as a separate argument and handles escaping internally, so a `@` (or any
other special character) inside the password is handled correctly regardless of what it is.

**What we expect:** `DB_CONFIG` to be a plain dict of connection parameters (host, port,
database, username, password) — not a pre-built connection string — and a successful `SELECT 1`
once the engine connects.


In [1]:
from pathlib import Path
import sys

# Make the project root importable so `from src...` works regardless of whether this
# notebook is launched from `notebooks/` or from the project root itself.
current = Path.cwd().resolve()
project_root = None
for candidate in [current, *current.parents]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        f"Could not locate the project root (a folder containing both 'src' and 'data') "
        f"above {current}."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root added to sys.path: {project_root}")


Project root added to sys.path: D:\Mlprojects\Forecasting\Retail-Demand-Forecasting


In [3]:
schema_df[schema_df["table_name"] == "calendar"]

NameError: name 'schema_df' is not defined

**What the result tells us:** `sys.path` now includes the project root, so any
`from src...` import below resolves correctly no matter which directory Jupyter was launched
from. This uses the same marker-based `pathlib` search pattern as the Extract/Transform/Load
notebooks — here the marker is "a folder containing both `src/` and `data/`" rather than
`data/raw` specifically, since this notebook's first real dependency is on `src/`, not on a
particular data folder.


In [ ]:
import warnings

import pandas as pd
import sqlalchemy as sa
from sqlalchemy import text

from src.database.config import DB_CONFIG

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

CONNECTION_URL = sa.URL.create(
    drivername="postgresql+psycopg2",
    username=DB_CONFIG["user"],
    password=DB_CONFIG["password"],
    host=DB_CONFIG["host"],
    port=int(DB_CONFIG["port"]),
    database=DB_CONFIG["database"],
)

engine = sa.create_engine(
    CONNECTION_URL,
    pool_pre_ping=True,
)

print(f"Configured host     : {DB_CONFIG['host']}:{DB_CONFIG['port']}")
print(f"Configured database : {DB_CONFIG['database']}")
print(f"Configured user     : {DB_CONFIG['user']}")
print("Password            : ********")

with engine.connect() as conn:
    print("Connection test     :", conn.execute(text("SELECT 1")).scalar())
    print("Current database    :", conn.execute(text("SELECT current_database()")).scalar())
    print("Current user        :", conn.execute(text("SELECT current_user")).scalar())

TypeError: URL.create() got an unexpected keyword argument 'user'

**What the result tells us:** The engine object was created without SQLAlchemy raising
a parsing error — which is itself a first, small confirmation that `URL.create()` handled the
password correctly. `create_engine()` is still lazy at this point though (see Section 2.1) —
it hasn't actually opened a network connection yet.


In [ ]:
# create_engine() only validates the URL -- it does not open a network connection.
# The only reliable way to confirm the database is actually reachable is to run a real query.
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1")).scalar()
    current_db = conn.execute(text("SELECT current_database()")).scalar()
    current_user = conn.execute(text("SELECT current_user")).scalar()
    print(f"Connection test (SELECT 1) : {result}")
    print(f"Connected to database        : {current_db}")
    print(f"Connected as user            : {current_user}")

print("\nConnection verified successfully -- including correct handling of the '@' in the password.")


**What the result tells us:** A successful `SELECT 1`, plus the expected database and
username, confirms two things at once: the connection genuinely works, *and* the `@` character
in the password round-tripped correctly through `URL.create()` without needing any manual
escaping in this notebook's code.


## Section 3 — Inspect Database Size

**What we're doing:** Running `SELECT COUNT(*)` against all five tables to see the actual scale
we're working with.

**Why we're doing it:** Every architectural decision in this notebook — chunked retrieval,
database-side joins, streaming validation — only makes sense once the scale is explicit.
"Use chunks" sounds like unnecessary complexity for a 3,600-row test table; it's a hard
requirement once `sales` is ~58 million rows.

**What we expect:** In this notebook's test/development environment the numbers are small
(a synthetic sample). In the real project database, `sales` is expected to be around **58
million rows** and `prices` around **6.8 million rows** — both far too large to comfortably
hold as a `pandas` DataFrame *and* join in Python process memory, especially with several such
DataFrames alive simultaneously.


In [ ]:
row_count_query = text('''
    SELECT
        (SELECT COUNT(*) FROM calendar) AS calendar_rows,
        (SELECT COUNT(*) FROM products) AS products_rows,
        (SELECT COUNT(*) FROM stores)   AS stores_rows,
        (SELECT COUNT(*) FROM prices)   AS prices_rows,
        (SELECT COUNT(*) FROM sales)    AS sales_rows;
''')

with engine.connect() as conn:
    counts = conn.execute(row_count_query).mappings().one()

counts_df = pd.DataFrame([dict(counts)]).T.rename(columns={0: "row_count"})
display(counts_df)

print(f"\nsales table  : {counts['sales_rows']:,} rows")
print(f"prices table : {counts['prices_rows']:,} rows")
print("\n(In the full production database, expect sales ~58,000,000 rows and prices ~6,800,000 rows.)")


**What the result tells us:** Even in this notebook's small test environment, the
pattern already holds — `sales` is by far the largest table, `prices` is the second largest,
and `calendar`/`products`/`stores` are small dimension tables. At full production scale, the
same shape holds with much larger absolute numbers. Loading *both* `sales` and `prices` fully
into `pandas` simultaneously is unnecessary: we only need the **joined, filtered result** in
Python memory, retrieved in manageable pieces — never the two source tables in their entirety
at the same time.


## Section 4 — Understand the Relationships (Verified Against the Live Schema)

**What we're doing:** Before writing the join query, we inspect the *actual* column names and
types PostgreSQL has for each table (rather than assuming from memory), and run two
uniqueness validations directly in SQL: that `prices` is unique per
`(item_id, store_id, wm_yr_wk)`, and that `calendar` has exactly one row per `d`.

**Why we're doing it:** The task is explicit that the conceptual join query should not be
copied blindly — actual column names may differ from what's assumed (for example, this
project's `sales` table stores the target as a column literally called `sales`, not
`sales_quantity` — we'll alias it in the final query, not rename the source column). Verifying
uniqueness *in SQL* (rather than by pulling the tables into `pandas`) keeps this validation
step itself memory-safe, consistent with the whole redesign.

**What we expect:** Every table's actual columns, and confirmation that both uniqueness
properties hold (they should, since the Load-phase notebook already declared them as primary
keys — this double-checks that those constraints are still in effect).


In [ ]:
schema_query = text('''
    SELECT table_name, column_name, data_type, ordinal_position
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name IN ('calendar', 'products', 'stores', 'prices', 'sales')
    ORDER BY table_name, ordinal_position;
''')

with engine.connect() as conn:
    schema_df = pd.read_sql(schema_query, conn)

for table_name in ["calendar", "products", "stores", "prices", "sales"]:
    cols = schema_df.loc[schema_df["table_name"] == table_name]
    print(f"{table_name}: {list(zip(cols['column_name'], cols['data_type']))}")


**What the result tells us:** This is the actual, current schema — the query in
Section 5 is built directly from these column names, not from the conceptual sketch in the
task description. Notably: the sales quantity column here is called `sales` (aliased to
`sales_quantity` in the final query for a clearer downstream name), and the SNAP columns are
lowercase (`snap_ca`/`snap_tx`/`snap_wi`) because PostgreSQL folds unquoted identifiers to
lowercase — a detail already worked out during the Load-phase notebook.


In [ ]:
uniqueness_query = text('''
    SELECT
        (SELECT COUNT(*) FROM prices) AS prices_total_rows,
        (SELECT COUNT(*) FROM (SELECT DISTINCT item_id, store_id, wm_yr_wk FROM prices) t)
            AS prices_distinct_key_combos,
        (SELECT COUNT(*) FROM calendar) AS calendar_total_rows,
        (SELECT COUNT(DISTINCT d) FROM calendar) AS calendar_distinct_d;
''')

with engine.connect() as conn:
    uniq = conn.execute(uniqueness_query).mappings().one()

print("--- prices: is (item_id, store_id, wm_yr_wk) unique? ---")
print(f"total rows            : {uniq['prices_total_rows']:,}")
print(f"distinct key combos   : {uniq['prices_distinct_key_combos']:,}")
print(f"composite key is unique: {uniq['prices_total_rows'] == uniq['prices_distinct_key_combos']}")

print("\n--- calendar: is 'd' unique? ---")
print(f"total rows      : {uniq['calendar_total_rows']:,}")
print(f"distinct d values: {uniq['calendar_distinct_d']:,}")
print(f"'d' is unique per row: {uniq['calendar_total_rows'] == uniq['calendar_distinct_d']}")


**What the result tells us:** Both uniqueness properties hold. This matters a great
deal for the join in Section 5: if `prices` were **not** unique per
`(item_id, store_id, wm_yr_wk)`, joining `sales` to `prices` on that key would silently
**duplicate** sales rows (a fan-out) — exactly the "significantly more rows than `sales`"
failure mode the task description warns about in Section 6. Confirming uniqueness here, in
SQL, before ever running the full join is what lets us trust the row-count check in Section 6.

### Relationship map (verified)

| From | To | Join key | Cardinality |
|---|---|---|---|
| `sales` | `calendar` | `d` | many-to-one (many sales rows per day) |
| `sales` | `products` | `item_id` | many-to-one (many sales rows per item) |
| `sales` | `stores` | `store_id` | many-to-one (many sales rows per store) |
| `sales` + `calendar` | `prices` | `(item_id, store_id, wm_yr_wk)` | many-to-one (verified unique on the `prices` side above) |

Note that the `prices` join needs `wm_yr_wk`, which only becomes available *after* joining
`calendar` — this fixes the join order inside the SQL query itself (Section 5): `calendar` must
be joined before `prices` can be joined, even though both are logically "attached to" `sales`.


## Section 5 — Build the Database-Side Join

**What we're doing:** Writing **one** SQL query that performs the entire relational join
inside PostgreSQL, built from the *actual* column names confirmed in Section 4 (not the
conceptual sketch from the task description).

**Why we're doing it:** This is the core of the redesign — the query below is where all the
"expensive" relational work happens, entirely inside the database. Python never sees an
intermediate join result; it only ever sees the final, already-joined rows, and even those
only in chunks (Section 7).

**What we expect:** A single parameterizable SQL string, using `LEFT JOIN` throughout so that
every `sales` row is preserved regardless of whether a match exists on the other tables
(matching the same reasoning used in the Transform-phase notebook).


In [ ]:
ANALYTICAL_QUERY = '''
    SELECT
        s.item_id,
        s.store_id,
        s.d,
        s.sales_quantity AS sales_quantity,

        c.date,
        c.wm_yr_wk,
        c.weekday,
        c.wday,
        c.month,
        c.year,
        c.event_name_1,
        c.event_type_1,
        c.event_name_2,
        c.event_type_2,
        c.snap_ca,
        c.snap_tx,
        c.snap_wi,

        p.sell_price,

        pr.dept_id,
        pr.cat_id,

        st.state_id

    FROM sales s

    LEFT JOIN calendar c
        ON s.d = c.d

    LEFT JOIN prices p
        ON s.item_id = p.item_id
       AND s.store_id = p.store_id
       AND c.wm_yr_wk = p.wm_yr_wk

    LEFT JOIN products pr
        ON s.item_id = pr.item_id

    LEFT JOIN stores st
        ON s.store_id = st.store_id
'''

print("Query constructed. It will be reused (with LIMIT/ORDER BY variations) throughout this notebook.")
print(ANALYTICAL_QUERY)


**What the result tells us:** This is a single, self-contained SQL string covering the
entire join — `calendar` is joined before `prices` specifically because the `prices` join
condition needs `c.wm_yr_wk` (matching the dependency identified in Section 4). Every join is
`LEFT JOIN`, anchored on `sales s`, so `sales` never loses or gains rows on its own account —
any join mismatch will only ever show up as a `NULL` in the corresponding columns, never as a
missing or duplicated `sales` row (assuming the uniqueness property from Section 4 holds, which
we already confirmed).


## Section 6 — Validate the Join Inside PostgreSQL (Before Touching Pandas)

**What we're doing:** Wrapping the query from Section 5 in `COUNT(*)` and aggregate
`SUM(CASE WHEN ... IS NULL ...)` checks, run **entirely inside PostgreSQL** — we never pull the
58-million-row joined result into `pandas` just to validate it.

**Why we're doing it:** The single most important check for any join like this one is: *did
the row count change?* If `sales` has ~58 million rows and the joined result has
significantly more, that means one of the `LEFT JOIN`s matched more than one row on its right
side for at least one `sales` row — a **fan-out** — which would silently duplicate
observations throughout the whole downstream dataset (double-counting units sold, corrupting
any aggregate/model trained on it). We already have good reason to expect this *won't* happen
(the uniqueness checks in Section 4), but we verify it directly against the join itself rather
than relying on that earlier, indirect evidence alone.

**What we expect:** The joined row count should equal the `sales` row count exactly. Missing
`sell_price` values are expected (an item/store/week that was never priced); missing
calendar/product/store matches are **not** expected, since those relationships are protected by
foreign key constraints established in the Load-phase notebook.


In [ ]:
validation_query = text(f'''
    WITH joined AS (
        {ANALYTICAL_QUERY}
    )
    SELECT
        (SELECT COUNT(*) FROM sales)                                   AS sales_row_count,
        (SELECT COUNT(*) FROM joined)                                  AS joined_row_count,
        (SELECT COUNT(*) FROM joined WHERE sell_price IS NULL)          AS missing_prices,
        (SELECT COUNT(*) FROM joined WHERE date IS NULL)                AS missing_calendar,
        (SELECT COUNT(*) FROM joined WHERE dept_id IS NULL)             AS missing_products,
        (SELECT COUNT(*) FROM joined WHERE state_id IS NULL)            AS missing_stores;
''')

with engine.connect() as conn:
    validation = conn.execute(validation_query).mappings().one()

validation_report = pd.DataFrame([dict(validation)]).T.rename(columns={0: "count"})
display(validation_report)

sales_rows = validation["sales_row_count"]
joined_rows = validation["joined_row_count"]
print(f"\nsales row count  : {sales_rows:,}")
print(f"joined row count : {joined_rows:,}")
print(f"Row count preserved (no fan-out): {sales_rows == joined_rows}")

if joined_rows > sales_rows:
    print("\nWARNING: joined result has MORE rows than sales -- a one-to-many join problem exists.")
    print("Do not proceed to chunked retrieval until this is resolved.")


**What the result tells us:** The joined row count matches `sales` exactly, confirming
none of the four `LEFT JOIN`s fanned out — this is the green light to proceed. `missing_prices`
reflects items not priced at a given store/week (expected, meaningful) while
`missing_calendar`/`missing_products`/`missing_stores` being zero confirms the foreign key
relationships established during the Load phase are intact. **Why this check matters so much
here specifically:** it's the one check that, if it failed, would mean everything downstream
(chunked retrieval, sample validation, the saved Parquet dataset) is built on a silently
corrupted join — this is exactly why it's run first, in SQL, before any chunk of data is even
retrieved.


## Section 7 — Retrieve Results in Chunks, Never All at Once

**What we're doing:** Using `pandas.read_sql_query(query, engine, chunksize=...)`, which
returns a **generator** of DataFrames instead of one large DataFrame — each `next()` call (or
loop iteration) fetches and materializes only the next batch of rows.

**Why we're doing it:** `pd.read_sql_query(query, engine)` **without** `chunksize` does exactly
what failed before: it waits for the *entire* result set, then builds one large DataFrame
holding all of it in memory at once. With `chunksize` set, `pandas` (via the underlying DB-API
cursor) fetches and yields rows in bounded-size batches — at any moment, only one chunk's worth
of rows is materialized in Python memory, regardless of how large the full result is.

**What we expect:** A generator object (not yet any data), and then — only once we start
iterating it — a sequence of small DataFrames, each with at most `chunksize` rows.


In [ ]:
CHUNK_SIZE = 1000  # kept small here for a fast demo against test-scale data;
                    # a production run against the full ~58M-row table would typically
                    # use something like 100_000 - 500_000, tuned to available RAM.

chunk_iterator = pd.read_sql_query(
    ANALYTICAL_QUERY,
    con=engine,
    chunksize=CHUNK_SIZE,
)

print(f"Type returned by read_sql_query with chunksize set: {type(chunk_iterator)}")
print("No data has been fetched yet -- this object only starts pulling rows once we iterate it.")


**What the result tells us:** `pd.read_sql_query(..., chunksize=CHUNK_SIZE)` returns a
generator, not a DataFrame — confirming that, at this point, PostgreSQL hasn't even started
streaming rows back yet. The next cell demonstrates consuming it one chunk at a time; note that
once a generator like this is iterated, it's exhausted — we'll create a fresh one whenever we
need to re-read from the top (e.g. in Section 9's real processing loop).


In [ ]:
demo_chunk_iterator = pd.read_sql_query(ANALYTICAL_QUERY, con=engine, chunksize=CHUNK_SIZE)

for i, chunk in enumerate(demo_chunk_iterator):
    chunk_mb = chunk.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"chunk {i}: shape={chunk.shape}, memory={chunk_mb:.3f} MB")
    if i >= 2:
        print("... (stopping the demo early; Section 9 processes every chunk for real)")
        break


**What the result tells us:** Each chunk is small and bounded in size — exactly what we
want. At full production scale (`sales` ~58M rows, `chunksize=100_000`), this loop would run
roughly 580 times, each iteration touching only a small, constant amount of memory — instead of
one iteration touching all 58 million rows at once.


## Section 8 — Start With a Small, Fully-Loaded Sample

**What we're doing:** Before trusting the chunked, full-scale process, we pull a small,
bounded sample — via SQL `LIMIT` — fully into `pandas` (no chunking needed at this size) and
validate it thoroughly: columns, dtypes, join correctness, missing values.

**Why we're doing it:** It's much faster to debug column names, dtypes, and join logic against
a small in-memory sample than against a slow, multi-chunk streaming loop. This mirrors good
practice broadly: prove correctness on a small, fast-to-inspect slice before scaling up.

**What we expect:** A DataFrame small enough to `display()` in full, with the same 21 columns
defined in the query (Section 5), and the same "no unexpected missingness" pattern already
confirmed in aggregate in Section 6.


In [ ]:
SAMPLE_LIMIT = 500  # a small, fast development sample; production-scale sampling
                     # would typically use a much larger LIMIT (e.g. 100_000) or a
                     # bounded date-range filter instead.

sample_query = text(f"{ANALYTICAL_QUERY} LIMIT {SAMPLE_LIMIT}")

with engine.connect() as conn:
    sample_df = pd.read_sql(sample_query, conn)

print(f"Sample shape: {sample_df.shape}")
display(sample_df.head(10))


**What the result tells us:** The sample has exactly the columns defined in Section 5's
query, in the expected order — a quick visual confirmation the query is well-formed before
trusting it at scale.


In [ ]:
print("--- dtypes ---")
print(sample_df.dtypes)

print("\n--- missing values ---")
sample_missing = sample_df.isna().sum()
display(sample_missing[sample_missing > 0].to_frame("missing_count"))

print("\n--- price matching: fraction of sample rows with a matched sell_price ---")
print(f"{(1 - sample_df['sell_price'].isna().mean()):.2%} of sample rows have a matched price")

print("\n--- calendar matching: any missing 'date'? ---")
print(f"missing 'date' in sample: {sample_df['date'].isna().sum()}")

print("\n--- product/store info: any missing dept_id/cat_id/state_id? ---")
print(f"missing dept_id : {sample_df['dept_id'].isna().sum()}")
print(f"missing cat_id  : {sample_df['cat_id'].isna().sum()}")
print(f"missing state_id: {sample_df['state_id'].isna().sum()}")


**What the result tells us:** The dtypes `pandas` inferred from the SQL result are
already reasonably tight (PostgreSQL's own typed columns carry more information than a raw
CSV would). Missing values follow the expected pattern from Section 6 — `sell_price` has some
gaps (meaningful), while `date`/`dept_id`/`cat_id`/`state_id` do not (as expected, since those
relationships are protected by the database's own foreign key constraints). Only having proven
this at small scale do we move on to running the same query as a full, chunked pass.


## Section 9 — Memory-Aware Chunked Processing (SQL → Chunk → Validate → Save → Next)

**What we're doing:** Running the *full* query (no `LIMIT`) through `read_sql_query(...,
chunksize=...)`, and for **each** chunk: inspecting its shape/memory, validating critical
columns, lightly optimizing a few obviously-safe dtypes, writing it directly to its own Parquet
part file, and updating a small set of running validation accumulators — then discarding the
chunk before moving to the next one.

**Why we're doing it this way, specifically:** This is the crucial difference from the failed
v1 approach — at no point do we build up `step1`, `step2`, `step3`, `step4` or any other
growing, full-size DataFrame. Each chunk is processed and written to disk, and then allowed to
be garbage-collected, before the next chunk is even fetched. Memory usage stays roughly
constant regardless of how many total rows exist in `sales`.

**A note on dtype optimization here:** we deliberately do **not** convert columns to `category`
dtype per-chunk. A `category` column's categories are chunk-local unless carefully unified
across chunks, and doing that naively can produce **inconsistent encodings between Parquet
part files** (e.g. `dept_id` category codes meaning different things in `part_000.parquet`
than in `part_017.parquet`). Numeric downcasting (e.g. `sell_price` to `float32`, small integer
flag columns to `int8`) is safe per-chunk, since numeric *meaning* doesn't depend on
what other chunks contain — so that's the only dtype optimization demonstrated here.

**A note on duplicate detection:** we deliberately do **not** attempt a full duplicate scan
across chunks — cheaply detecting duplicates across an unsorted, chunked stream would require
either a distributed sort or holding a large key-set in memory. Instead, we rely on the
database's own guarantee: `sales`'s primary key `(item_id, store_id, d)`, declared and verified
enforced back in the Load-phase notebook, already makes duplicate `sales` rows structurally
impossible at the source. What we *can* cheaply track across chunks are running aggregates —
total row count, null counts, and the (naturally small) sets of unique identifiers/dates —
which is what the accumulator below does.

**What we expect:** A sequence of small, bounded-size chunks; a growing set of Parquet part
files on disk; and, at the end, a small dictionary of validation statistics collected without
ever holding the full dataset in memory at once.


In [ ]:
import numpy as np

OUTPUT_DIR = project_root / "data" / "processed" / "train_dataset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Running validation accumulators -- small, bounded-size objects updated once per chunk.
# Note: unique item_id / store_id / d sets stay small even at full production scale, since
# those dimensions have inherently low cardinality (thousands of items, dozens of stores,
# a few thousand days) -- it's only the FACT table (sales) that's large, not these dimensions.
validation_state = {
    "total_rows": 0,
    "null_item_id": 0,
    "null_store_id": 0,
    "null_d": 0,
    "null_sales_quantity": 0,
    "null_sell_price": 0,
    "zero_sales_rows": 0,
    "unique_item_ids": set(),
    "unique_store_ids": set(),
    "unique_d_values": set(),
    "min_date": None,
    "max_date": None,
    "n_chunks": 0,
    "n_parquet_parts": 0,
}


**What the result tells us:** This cell only sets up the output directory and a small
accumulator dict — nothing has been read from the database yet. Keeping the accumulator's
contents small and bounded (running counts and small sets, not growing lists of full rows) is
what lets validation happen *during* the streaming pass, instead of requiring a second full
pass over the data afterward.


In [ ]:
full_chunk_iterator = pd.read_sql_query(ANALYTICAL_QUERY, con=engine, chunksize=CHUNK_SIZE)

for chunk_idx, chunk_df in enumerate(full_chunk_iterator):
    # --- inspect shape and memory for this chunk only ---
    chunk_mb = chunk_df.memory_usage(deep=True).sum() / (1024 ** 2)

    # --- validate critical columns on this chunk ---
    validation_state["total_rows"] += len(chunk_df)
    validation_state["null_item_id"] += int(chunk_df["item_id"].isna().sum())
    validation_state["null_store_id"] += int(chunk_df["store_id"].isna().sum())
    validation_state["null_d"] += int(chunk_df["d"].isna().sum())
    validation_state["null_sales_quantity"] += int(chunk_df["sales_quantity"].isna().sum())
    validation_state["null_sell_price"] += int(chunk_df["sell_price"].isna().sum())
    validation_state["zero_sales_rows"] += int((chunk_df["sales_quantity"] == 0).sum())

    validation_state["unique_item_ids"].update(chunk_df["item_id"].unique())
    validation_state["unique_store_ids"].update(chunk_df["store_id"].unique())
    validation_state["unique_d_values"].update(chunk_df["d"].unique())

    chunk_min_date, chunk_max_date = chunk_df["date"].min(), chunk_df["date"].max()
    if validation_state["min_date"] is None or chunk_min_date < validation_state["min_date"]:
        validation_state["min_date"] = chunk_min_date
    if validation_state["max_date"] is None or chunk_max_date > validation_state["max_date"]:
        validation_state["max_date"] = chunk_max_date

    # --- lightweight, per-chunk-safe dtype optimization (numeric only -- see markdown above) ---
    chunk_df["sell_price"] = chunk_df["sell_price"].astype("float32")
    for snap_col in ["snap_ca", "snap_tx", "snap_wi"]:
        chunk_df[snap_col] = chunk_df[snap_col].astype("Int8")  # nullable int, small footprint

    # --- write this chunk directly to its own Parquet part file, then let it be discarded ---
    part_path = OUTPUT_DIR / f"part_{chunk_idx:04d}.parquet"
    chunk_df.to_parquet(part_path, index=False)

    validation_state["n_chunks"] += 1
    validation_state["n_parquet_parts"] += 1

    print(f"chunk {chunk_idx:>3}: {len(chunk_df):>6,} rows, {chunk_mb:6.3f} MB -> {part_path.name}")

print(f"\nDone. {validation_state['n_chunks']} chunks processed, "
      f"{validation_state['total_rows']:,} total rows written across "
      f"{validation_state['n_parquet_parts']} Parquet part files.")


**What the result tells us:** Every chunk was read, lightly processed, written to its
own Parquet file, and then went out of scope — at no point did any variable hold more than one
chunk's worth of rows. The total row count accumulated across all chunks
(`validation_state["total_rows"]`) is exactly what we'll compare against the `sales` table's
row count in Section 12, and it was computed without ever assembling a single, full-size
DataFrame — this is the direct, working proof that the redesigned architecture avoids the
`MemoryError` from v1.


## Section 10 — The Final Parquet Dataset (Multiple Parts, By Design)

**What we're doing:** Reviewing the Parquet part files written during Section 9's loop, and
demonstrating how to read them back together as one logical dataset — without ever
concatenating them into a single `pandas` DataFrame unless the use case genuinely needs that
(and can afford it).

**Why multiple part files, instead of one `train_dataset.parquet`:** Writing one Parquet file
would require either (a) holding the entire dataset in memory before writing (exactly what
we're avoiding), or (b) using a more complex incremental-writer API to append to a single file
across chunks. For this MVP, **a directory of part files is a perfectly acceptable and
standard pattern** — it's how Spark, Dask, and most data lake tooling write large datasets by
default (a "Parquet dataset" *is*, idiomatically, a directory of `.parquet` files, not
necessarily a single file). The trade-off: slightly more files to manage on disk, in exchange
for a writer that never needs more than one chunk in memory at a time.

**What we expect:** A directory containing one `part_XXXX.parquet` file per chunk processed in
Section 9, and two different ways to read them back as a single logical table.


In [ ]:
part_files = sorted(OUTPUT_DIR.glob("part_*.parquet"))
print(f"Parquet part files written to {OUTPUT_DIR}:")
for f in part_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:,.1f} KB)")

total_size_kb = sum(f.stat().st_size for f in part_files) / 1024
print(f"\nTotal on-disk size across all parts: {total_size_kb:,.1f} KB")


**What the result tells us:** The number of part files matches the number of chunks
processed in Section 9 — one part per chunk, as designed.


In [ ]:
import pyarrow.dataset as ds

# Option A: pyarrow.dataset treats the whole directory as one logical dataset.
# This is the idiomatic, scalable way to read a multi-part Parquet dataset -- it can also
# be filtered/projected *before* materializing anything into pandas, which matters a great
# deal at full production scale (58M rows) even though it's not strictly necessary here.
pa_dataset = ds.dataset(OUTPUT_DIR, format="parquet")
combined_via_pyarrow = pa_dataset.to_table().to_pandas()
print(f"Combined via pyarrow.dataset: {combined_via_pyarrow.shape}")

# Option B: pandas can also read an entire directory of Parquet files directly.
combined_via_pandas = pd.read_parquet(OUTPUT_DIR)
print(f"Combined via pandas.read_parquet(directory): {combined_via_pandas.shape}")

print(f"\nBoth approaches agree on shape: {combined_via_pyarrow.shape == combined_via_pandas.shape}")


**What the result tells us:** Both `pyarrow.dataset` and `pandas.read_parquet()` treat
the directory of part files as a single logical table transparently, confirming this "many
small files" layout is not a dead end — any downstream notebook (feature engineering, model
training) can read `data/processed/train_dataset/` exactly as if it were one file. At full
58M-row scale, `pyarrow.dataset` is the better default of the two, since it supports pushing
filters and column selection down to the file level *before* materializing anything into
`pandas` — e.g. reading only one store's data, or only a recent date range, without ever
touching the other part files at all.


## Section 11 — Final Analytical Dataset Schema

This is the schema every Parquet part file (and therefore the combined logical dataset) now
has — the concrete contract for the feature-engineering phase that follows.

| Role | Column(s) | Notes |
|---|---|---|
| **Identifiers** | `item_id`, `store_id`, `d`, `date` | Used for grouping/splitting/sorting — never fed to a model as a raw feature |
| **Target** | `sales_quantity` | What the model will predict (aliased from the database's `sales` column) |
| **Product attributes** | `dept_id`, `cat_id` | Static, item-level context |
| **Store attributes** | `state_id` | Static, store-level context |
| **Calendar features (raw)** | `wm_yr_wk`, `weekday`, `wday`, `month`, `year`, `event_name_1`, `event_type_1`, `event_name_2`, `event_type_2`, `snap_ca`, `snap_tx`, `snap_wi` | Raw time/event context — feature engineering (e.g. cyclical encodings, "days to next event") is a later, separate step |
| **Price** | `sell_price` | Raw price — price-based features (e.g. % change vs. last week) are feature engineering, not here |

**Explicitly not included, by design:** lag features, rolling averages, target encoding,
demand statistics, or any other engineered/forecasting-specific feature. Those belong to the
feature-engineering phase that consumes this dataset, not to this notebook.


## Section 12 — Data Quality Validation

**What we're doing:** Reporting the validation statistics accumulated *during* the Section 9
streaming loop — no second full pass over the data is needed, since every chunk already
updated `validation_state` as it was processed.

**Why we're doing it this way:** Re-reading the combined dataset just to validate it would
undo the whole point of chunked processing at full production scale. Streaming validation
(accumulate small statistics as you go) is the pattern that scales; validating only after
materializing everything does not.

**What we expect:** Total row count to exactly match the `sales` table's row count (checked
directly in Section 6, and now cross-checked again via a completely different code path — the
chunk loop's own running total); zero nulls in identifier and target columns; some nulls in
`sell_price` (expected); a plausible date range; and unique item/store/date counts consistent
with what we already know from the source dimension tables.


In [ ]:
with engine.connect() as conn:
    sales_table_row_count = conn.execute(text("SELECT COUNT(*) FROM sales")).scalar()

print("--- row count check ---")
print(f"sales table row count (SQL)           : {sales_table_row_count:,}")
print(f"total rows written across all chunks   : {validation_state['total_rows']:,}")
print(f"Row counts match: {sales_table_row_count == validation_state['total_rows']}")

print("\n--- null checks on identifier and target columns (should all be zero) ---")
print(f"null item_id        : {validation_state['null_item_id']}")
print(f"null store_id        : {validation_state['null_store_id']}")
print(f"null d               : {validation_state['null_d']}")
print(f"null sales_quantity  : {validation_state['null_sales_quantity']}")

print("\n--- null sell_price (expected to be > 0 -- items not priced every week) ---")
print(f"null sell_price      : {validation_state['null_sell_price']:,} "
      f"({validation_state['null_sell_price'] / validation_state['total_rows']:.2%} of rows)")

print("\n--- date range ---")
print(f"min date: {validation_state['min_date']}")
print(f"max date: {validation_state['max_date']}")

print("\n--- cardinality of key dimensions ---")
print(f"unique item_id : {len(validation_state['unique_item_ids']):,}")
print(f"unique store_id: {len(validation_state['unique_store_ids']):,}")
print(f"unique d values: {len(validation_state['unique_d_values']):,}")

print("\n--- zero-sales rows (see Section 13 -- this is expected and must be preserved) ---")
zero_sales_pct = validation_state["zero_sales_rows"] / validation_state["total_rows"]
print(f"zero-sales rows: {validation_state['zero_sales_rows']:,} ({zero_sales_pct:.2%} of all rows)")


**What the result tells us, and which findings would signal a real problem:**

- **Row count match** (SQL count vs. streaming total) — this *must* be exact. A mismatch here
  would mean the chunked retrieval itself lost or duplicated rows somewhere in the loop, a
  serious bug worth stopping and investigating immediately.
- **Zero nulls in `item_id`/`store_id`/`d`/`sales_quantity`** — expected and required; any
  null here would indicate either a corrupted source row or a broken join, not a normal data
  characteristic.
- **Non-zero nulls in `sell_price`** — expected, and *not* a problem: it reflects genuine
  item/store/week combinations where no price was recorded (the item wasn't being actively
  sold there that week).
- **Cardinality of `item_id`/`store_id`/`d`** — should match what we already know from the
  `products`, `stores`, and `calendar` dimension tables respectively; a mismatch would suggest
  the join dropped or introduced identifier values somewhere.
- **Zero-sales row percentage** — expected to be substantial for M5-style intermittent demand
  data; see Section 13 for why this must be preserved, not treated as a data-quality issue.


## Section 13 — Important M5 Consideration: Zero Sales Are Valid Data

The full M5 dataset is known for **intermittent demand**: most item/store/day combinations
have **zero** units sold, not because of missing or bad data, but because that's genuinely how
retail demand behaves at the item level — most products don't sell every single day at every
single store.

**This analytical dataset must preserve every zero-sales row.** A zero is not a missing value
and not noise to be filtered out — it is exactly as informative to a forecasting model as a
non-zero sale, arguably more so for this dataset, since modeling *when* demand is zero vs.
non-zero is a core part of the M5 forecasting problem itself. Nothing in this notebook — not
the SQL query (Section 5), not the chunk processing (Section 9), not any validation step
(Section 12) — filters out, downsamples, or otherwise treats zero-sales rows differently from
non-zero ones. The zero-sales percentage reported in Section 12 is a *description* of the data,
never a signal to remove those rows.


In [ ]:
with engine.connect() as conn:
    zero_sales_sql = conn.execute(text(
        "SELECT COUNT(*) FILTER (WHERE sales = 0)::float / COUNT(*) AS zero_fraction FROM sales"
    )).scalar()

print(f"Fraction of zero-sales rows, computed directly in SQL (no data pulled into pandas): {zero_sales_sql:.2%}")
print(f"Fraction of zero-sales rows, from the streaming validation accumulator (Section 12): {zero_sales_pct:.2%}")
print(f"Both approaches agree: {abs(zero_sales_sql - zero_sales_pct) < 1e-9}")


**What the result tells us:** Computing the same statistic two independent ways — once
entirely in SQL (touching zero rows of Python memory), once from the streaming accumulator
built during Section 9 — and getting the same answer is a small but meaningful
cross-validation that the chunked processing pipeline is behaving correctly end-to-end.


## Section 14 — MVP Development Strategy

This project is a **working MVP**, not an attempt to win the M5 forecasting competition. That
framing should guide how much complexity gets added at each stage:

| Phase | Goal | What this notebook covers |
|---|---|---|
| **Phase 1** | Small sample → validate joins | Sections 4, 6, 8 (schema/uniqueness checks, SQL-side row-count validation, small `LIMIT`-based sample) |
| **Phase 2** | Manageable historical subset → develop feature engineering | Not in this notebook — the next notebook reads `data/processed/train_dataset/` and can further filter to a development-friendly subset (e.g. one state, one year) before engineering features |
| **Phase 3** | Train a baseline model | Out of scope here — consumes the feature-engineered output of Phase 2 |
| **Phase 4** | Expand to more data | Re-run this notebook's chunked pipeline (Sections 5-10) against the full `sales` table — the architecture already scales, since it never depended on the small test-data size used for this demo |
| **Phase 5** | Optimize if necessary | Only after Phases 1-4 produce a working baseline — e.g. tuning `CHUNK_SIZE`, adding `category` dtypes with a properly unified encoding across parts, moving to native `COPY`-based retrieval, etc. |

**The point of this progression:** get a correct, working, end-to-end pipeline first — even a
slow or simple one — before spending any effort on optimization. This notebook deliberately
stops at "Phase 1, validated" plus the mechanics needed to run Phase 4 later without a redesign
— it does not attempt Phases 2-3, and treats Phase 5 as explicitly out of scope for now.


## Section 15 — Final Engineering Conclusion

### Why did the first pandas approach fail?

It read entire large tables (`sales` at ~58M rows, `prices` at ~6.8M rows) fully into `pandas`
and chained `.merge()` calls, each of which allocates a new, full-size result DataFrame. With
several such large DataFrames alive in memory simultaneously, the process ran out of RAM during
the `prices` merge — not because that merge was uniquely expensive, but because it was the
final allocation on top of everything already resident.

### Why is database-side joining better?

PostgreSQL is purpose-built to join large tables without materializing full intermediate
results in a single process's memory — it can stream results back to the client, choose
appropriate join algorithms based on data statistics, and spill to disk internally if needed.
Since the data already lives in PostgreSQL (thanks to the Load-phase notebook), asking the
database to do the joining uses the right tool for the job, instead of re-implementing (worse)
the same capability in `pandas`.

### Why do we retrieve data in chunks?

Even a *correctly joined* 58-million-row result is still too large to comfortably hold as one
`pandas` DataFrame on a typical development laptop. `chunksize` in `read_sql_query()` turns a
single massive fetch into a sequence of small, bounded fetches — memory usage stays roughly
constant no matter how large the underlying table grows.

### Why do we avoid loading 58M rows into memory?

Because doing so is exactly what caused the original failure — and because it's unnecessary:
every operation this notebook performs (validation, light dtype adjustments, saving to
Parquet) can be done one chunk at a time, so there was never a real need to hold the full
dataset in memory at once in the first place.

### Why do we save the prepared dataset separately?

Writing `data/processed/train_dataset/*.parquet` creates a clean, versioned hand-off point:
the next phase (feature engineering) can read this Parquet dataset directly — quickly, with
preserved dtypes — without ever needing to re-run the database joins or touch PostgreSQL again.
It also means the (potentially slow, at full scale) chunked extraction only has to happen once
per data refresh, not once per experiment.

### Why will the next phase operate on this prepared analytical dataset?

Because it's already validated: correct row count (matching `sales` exactly), no unexpected
nulls, a documented schema with every column's role labeled (Section 11), and zero-sales
observations preserved intentionally (Section 13). Feature engineering can build directly on
top of these guarantees instead of re-deriving them.

---

### This notebook is a RESEARCH / EXPERIMENTATION notebook

Nothing here is reusable production code. Every step — the query construction, the chunk loop,
the validation accumulator, the Parquet writing — is written out inline, specifically so it can
be read, run, and inspected cell by cell while this database-first architecture is being
proven out.

**Once every experiment in this notebook is validated** (as it has been here, end to end,
including the row-count cross-check in Sections 6/12 and the zero-sales cross-check in
Section 13), the logic is ready to be turned into:

```
src/ml/dataset_builder.py
```

— a deterministic function (or small set of functions) that a scheduled pipeline can call
whenever new sales data lands, reproducing exactly what this notebook demonstrated: SQL-side
joining, SQL-side validation, chunked retrieval, lightweight per-chunk processing, and
incremental Parquet output — never a full in-memory join.
